# Missing values

Real tables have blanks, and most scikit-learn models refuse to run on them. This notebook shows the two usual fixes: **drop** the rows, or **fill** the blanks with the mean or the median.

This notebook goes with the **Missing values** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

## 1. Load the data

`customers.csv` is a small made-up table of 80 shoppers: their **country**, **age** and **salary**, and whether they **purchased** (1) or not (0). It was generated for this course, so it is free to use. A few ages and salaries are left blank on purpose.

In [ ]:
import numpy as np
import pandas as pd

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/customers.csv"
dataset = pd.read_csv(DATA_URL)
dataset.head()

## 2. Find the blanks

Pandas calls a blank `NaN`. `isna()` marks every blank and `sum()` counts them per column.

In [ ]:
print(dataset.isna().sum())
print('Rows with at least one blank:', dataset.isna().any(axis=1).sum(), 'of', len(dataset))

## 3. Option A: drop the rows

Simple, but every row with a blank disappears, including the values that were there.

In [ ]:
dropped = dataset.dropna()
print(len(dataset), 'rows before,', len(dropped), 'rows after')
print('Mean age after dropping:   ', round(dropped['age'].mean(), 2))
print('Mean salary after dropping:', round(dropped['salary'].mean()))

## 4. Option B: fill with the mean or the median

`SimpleImputer` works out one value per column from the numbers that exist, then writes it into the blanks. Try `strategy='median'` too and compare.

In [ ]:
from sklearn.impute import SimpleImputer

numbers = dataset[['age', 'salary']].values
imputer = SimpleImputer(strategy='mean')   # or 'median'
filled = imputer.fit_transform(numbers)

print('Values used to fill [age, salary]:', imputer.statistics_.round(1))
print('Blanks left:', np.isnan(filled).sum(), '| rows kept:', len(filled))

## 5. Do it the careful way: split first

The fill values should come from the **training** rows only. Otherwise information from the test rows leaks into training. Fit the imputer on the training set, then use it on both sets. (The next lessons explain the split.)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(numbers, test_size=0.25, random_state=0)

imputer = SimpleImputer(strategy='median')
imputer.fit(X_train)                  # learn from the training rows only
X_train = imputer.transform(X_train)
X_test = imputer.transform(X_test)    # fill the test rows with the training numbers
print('Median age and salary of the training rows:', imputer.statistics_)

## Try it yourself

- Switch `strategy` to `'median'` in step 4. Which fill value changes more, age or salary?
- Add `strategy='most_frequent'`. When would that make sense?